In [ ]:
import math
import csv
from datetime import date
from os import path, makedirs

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from scipy import interpolate
from scipy.interpolate import make_interp_spline, BSpline
from scipy.spatial import ConvexHull, convex_hull_plot_2d
# from scipy.optimize import minimize
from scipy.optimize import LinearConstraint
from scipy.optimize import NonlinearConstraint
from scipy.optimize import Bounds
from cobyqa import minimize
from scipy.integrate import trapezoid
from collections import defaultdict

In [5]:
def read_xy_from_csv(csv_filepath, x_col=0, y_col=1, delimiter=','):
    """Read ordered x,y points from a CSV file.

    Parameters:
    csv_filepath: str
        Path to a CSV file with at least two columns of numeric data.
    x_col: int, optional
        Column index for x coordinates. Default is 0.
    y_col: int, optional
        Column index for y coordinates. Default is 1.
    delimiter: str, optional
        Field delimiter for the CSV file. Default is ','.

    Returns:
    x: numpy.ndarray
        Array of x coordinate values.
    y: numpy.ndarray
        Array of y coordinate values.
    """
    df = pd.read_csv(csv_filepath, header=None, comment='#', delimiter=delimiter)
    if df.shape[1] < 2:
        raise ValueError("CSV file must contain at least two columns of numeric x and y data.")
    df = df.iloc[:, [x_col, y_col]].apply(pd.to_numeric, errors='coerce')
    if df.isna().any().any():
        raise ValueError("CSV file contains non-numeric values in the x or y columns.")
    x = df.iloc[:, 0].to_numpy(dtype=float)
    y = df.iloc[:, 1].to_numpy(dtype=float)
    return x, y


def curve_length_from_csv(csv_filepath, n_eval=1000, spline_order=3,
                          x_col=0, y_col=1, delimiter=','):
    """Compute the length of a spline curve through ordered CSV points.

    The function reads x/y points from the provided CSV file, constructs a
    parametric spline curve that passes through the points in order, and
    integrates the curve length.

    Parameters:
    csv_filepath: str
        Path to the CSV file containing ordered x and y coordinates.
    n_eval: int, optional
        Number of points to evaluate along the spline for length estimation.
    spline_order: int, optional
        Order of the spline. Default is 3 (cubic) and is automatically reduced
        when there are too few data points.
    x_col: int, optional
        Index of the column containing x coordinates.
    y_col: int, optional
        Index of the column containing y coordinates.
    delimiter: str, optional
        Field delimiter used by the CSV file.

    Returns:
    float
        Estimated length of the interpolated curve.
    """
    x, y = read_xy_from_csv(csv_filepath, x_col=x_col, y_col=y_col,
                            delimiter=delimiter)
    if x.size < 2:
        return 0.0
    t = np.linspace(0.0, 1.0, x.size)
    k = min(spline_order, x.size - 1)
    spline = make_interp_spline(t, np.column_stack((x, y)), k=k, axis=0)
    t_eval = np.linspace(t[0], t[-1], n_eval)
    derivative = spline.derivative()
    dx_dy = derivative(t_eval)
    speeds = np.hypot(dx_dy[:, 0], dx_dy[:, 1])
    return float(trapezoid(speeds, t_eval))

In [6]:
csv_filepath = 'data/inner_63_stroke.csv'
result = curve_length_from_csv(csv_filepath, n_eval=1000, spline_order=3, x_col=0, y_col=1, delimiter=',')